In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.utils import load_subsample_indices
from src.preprocessing import (
    preprocess_data, EXCLUDED_COLUMNS, FULL_FEATURE_SET,
    CATEGORICAL_FEATURES, NUMERICAL_FEATURES, RARE_THRESHOLD,
)

plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 40)

DATA_PATH = "../data/hotel_bookings_course_release_v1.csv"
INDICES_PATH = "../data/subsample_indices_v1_n30000_seed12345.txt"

df_full = pd.read_csv(DATA_PATH)
indices = load_subsample_indices(INDICES_PATH)
df = df_full.iloc[indices].reset_index(drop=True)
#df = df_full

print(f"Full dataset : {df_full.shape}")
print(f"Subsample    : {df.shape}")


## 1. Subsample Representativeness


In [ ]:
key_nums = ["lead_time", "adr", "total_of_special_requests", "booking_changes"]
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
for ax, col in zip(axes.flat, key_nums):
    ax.hist(df_full[col].dropna(), bins=40, alpha=0.5, label="full", density=True)
    ax.hist(df[col].dropna(), bins=40, alpha=0.5, label="subsample", density=True)
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()
print("Subsample means vs full means:")
print(pd.DataFrame({
    "full": df_full[key_nums].mean(),
    "subsample": df[key_nums].mean(),
}).round(2))


## 2. Data Quality Snapshot

In [ ]:
snap = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_missing": df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(dropna=False),
    "pct_unique": (df.nunique(dropna=False) / len(df) * 100).round(2)
}).sort_values("pct_missing", ascending=False)
print(snap[snap["n_missing"] > 0])


In [ ]:
num_duplicates = df.duplicated().sum()
print(f'Duplicate rows: {num_duplicates}')

### Outlier Summary

In [ ]:
num_features = [c for c in NUMERICAL_FEATURES if c in df.columns]
num_df = df[num_features].copy()

q25  = num_df.quantile(0.25)
q75  = num_df.quantile(0.75)
iqr  = q75 - q25
p99  = num_df.quantile(0.99)
iqr_upper = q75 + 1.5 * iqr
n_outliers = (num_df > iqr_upper).sum()

outlier_summary = pd.DataFrame({
    "mean":      num_df.mean().round(2),
    "median":    num_df.median().round(2),
    "p25":       q25.round(2),
    "p75":       q75.round(2),
    "IQR":       iqr.round(2),
    "p99":       p99.round(2),
    "upper_fence": iqr_upper.round(2),
    "n_outliers (IQR rule)": n_outliers,
    "pct_outliers": (n_outliers / len(num_df) * 100).round(2),
}).sort_values("n_outliers (IQR rule)", ascending=False)

print("Outlier summary (IQR rule: > Q3 + 1.5*IQR):")
print(outlier_summary.to_string())


In [ ]:
n_num = len(num_features)
n_cols_box = 4
n_rows_box = -(-n_num // n_cols_box)
fig, axes = plt.subplots(n_rows_box, n_cols_box, figsize=(16, n_rows_box * 3))
for ax, col in zip(axes.flat, num_features):
    ax.boxplot(df[col].dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor='lightblue'))
    ax.set_title(col, fontsize=9)
    ax.tick_params(axis='x', labelbottom=False)
for ax in axes.flat[n_num:]:
    ax.set_visible(False)
plt.suptitle('Boxplots — numerical features (outliers visible above whiskers)')
plt.tight_layout()
plt.show()


## 3. Feature Distributions


In [ ]:
num_features = [c for c in NUMERICAL_FEATURES if c in df.columns]
n_cols = 3
n_rows = -(-len(num_features) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 3))
for ax, col in zip(axes.flat, num_features):
    ax.hist(df[col].dropna(), bins=40)
    ax.set_title(col)
for ax in axes.flat[len(num_features):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

skew = df[num_features].skew().sort_values(ascending=False)
print("Skewness (|>1| \u2192 consider robust scaling):")
print(skew.round(2))


In [ ]:
cat_features = [c for c in CATEGORICAL_FEATURES if c in df.columns]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flat, cat_features):
    counts = df[col].value_counts()
    ax.barh(counts.index, counts.values)
    ax.set_title(col)
for ax in axes.flat[len(cat_features):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

### Rare-Category Analysis

In [ ]:
print(f"Rare-category threshold: {RARE_THRESHOLD:.0%}\n")
for col in cat_features:
    freq = df[col].value_counts(normalize=True).round(4)
    rare = freq[freq < RARE_THRESHOLD]
    if not rare.empty:
        print(f"{col}: {len(rare)} rare categories \u2192 grouped into 'Other'")
        print(rare.to_string(), "\n")
    else:
        print(f"{col}: no rare categories\n")

## 4. Pre vs Post Winsorization

In [ ]:
from src.preprocessing import WINSOR_CONFIG

winsor_cols_present = [c for c in WINSOR_CONFIG if c in df.columns]
fig, axes = plt.subplots(2, len(winsor_cols_present), figsize=(14, 6))

for j, col in enumerate(winsor_cols_present):
    q = WINSOR_CONFIG[col]
    bound = float(np.quantile(df[col].dropna().values, q))
    winsorized = df[col].clip(upper=bound)

    axes[0, j].hist(df[col].dropna(), bins=40, color='steelblue', alpha=0.8)
    axes[0, j].axvline(bound, color='red', linestyle='--',
                       label=f'p{int(q*100)}={bound:.1f}')
    axes[0, j].set_title(f'{col}\n(before)')
    axes[0, j].legend(fontsize=7)

    axes[1, j].hist(winsorized, bins=40, color='coral', alpha=0.8)
    axes[1, j].set_title(f'{col}\n(after winsorization)')

plt.suptitle('Pre vs post winsorization — most skewed features (Lab2 Q9)')
plt.tight_layout()
plt.show()


## 5. Feature Correlation

In [ ]:
num_df = df[[c for c in NUMERICAL_FEATURES if c in df.columns]].copy()
corr = num_df.corr()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(corr)))
ax.set_yticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
plt.colorbar(im, ax=ax)
ax.set_title("Pearson correlation \u2014 numerical features")
plt.tight_layout()
plt.show()

pairs = (
    corr.abs()
    .where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)
print("Top correlated pairs:")
print(pairs.head(10).round(3))


### Duplicate and Near-Duplicate Column Analysis

In [ ]:
X_check, names_check = preprocess_data(df, feature_set='full', scaler='standard')
feat_df = pd.DataFrame(X_check, columns=names_check)

# 1. Exact duplicate columns
dup_mask = feat_df.T.duplicated()
dup_cols = feat_df.columns[dup_mask].tolist()
print(f'Exact duplicate columns after preprocessing: {len(dup_cols)}')
if dup_cols:
    print('  ->', dup_cols)
else:
    print('  -> None found')

# 2. Near-duplicate columns (|Pearson r| > 0.95)
corr_post = feat_df.corr().abs()
upper = corr_post.where(np.triu(np.ones(corr_post.shape), k=1).astype(bool))
near_dup = [
    (c, r, round(upper.loc[c, r], 4))
    for c in upper.columns
    for r in upper.index
    if pd.notna(upper.loc[c, r]) and upper.loc[c, r] > 0.95
]
print(f'\nNear-duplicate pairs (|r| > 0.95) after preprocessing: {len(near_dup)}')
for a, b, val in sorted(near_dup, key=lambda x: -x[2]):
    print(f'  {a}  <->  {b}  (r={val})')

# 3. Heatmap of high-correlation pairs only
high_corr_cols = list(set(
    [a for a, b, _ in near_dup] + [b for a, b, _ in near_dup] + dup_cols
))
if high_corr_cols:
    sub = feat_df[high_corr_cols].corr()
    sz = max(6, len(high_corr_cols))
    fig, ax = plt.subplots(figsize=(sz, sz - 1))
    im = ax.imshow(sub.values, vmin=-1, vmax=1, cmap='RdBu_r')
    ax.set_xticks(range(len(sub)))
    ax.set_yticks(range(len(sub)))
    ax.set_xticklabels(sub.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(sub.columns, fontsize=8)
    plt.colorbar(im, ax=ax)
    ax.set_title('Near-duplicate columns (|r| > 0.95) — post preprocessing')
    plt.tight_layout()
    plt.show()
else:
    print('\nNo near-duplicate columns to plot.')


## 6. PCA Scree Plot

In [ ]:
from sklearn.decomposition import PCA

X_pca, _= preprocess_data(df, feature_set='full', scaler='standard')
pca_full = PCA(random_state=0)
pca_full.fit(X_pca)

cum_var = np.cumsum(pca_full.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
            pca_full.explained_variance_ratio_, alpha=0.7)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot')

axes[1].plot(range(1, len(cum_var) + 1), cum_var, marker='o', markersize=4)
for threshold, ls in [(0.80, '--'), (0.85, '-'), (0.90, ':')]:
    n_comp = int(np.searchsorted(cum_var, threshold)) + 1
    axes[1].axhline(threshold, color='red', linestyle=ls, alpha=0.6,
                    label=f'{threshold:.0%} -> {n_comp} components')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Variance — PCA')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

for threshold in [0.80, 0.85, 0.90]:
    n = int(np.searchsorted(cum_var, threshold)) + 1
    print(f'{threshold:.0%} variance explained by {n} components')


## 7. Segmentation Time

In [ ]:
from src.preprocessing import EXCLUDED_COLUMNS, CONTEXT_BLOCK, FULL_FEATURE_SET

print("=== Excluded from clustering inputs ===")
for col in EXCLUDED_COLUMNS:
    print(f"  {col}")

print("\n=== Context features derived at booking creation ===")
for col in CONTEXT_BLOCK:
    print(f"  {col}")

print("\n=== Full feature set (clustering inputs) ===")
for col in FULL_FEATURE_SET:
    print(f"  {col}")


X_full, names_full = preprocess_data(df, feature_set="full", scaler="standard")
X_nocontext, names_nocontext = preprocess_data(df, feature_set="no_context", scaler="standard")
print(f"\nfull representation: {X_full.shape[1]} features")
print(f"no_context representation: {X_nocontext.shape[1]} features")
print(f"context columns in full: {[n for n in names_full if n in CONTEXT_BLOCK]}")

## 8. Distance Metric and Scaling Analysis


In [ ]:
from sklearn.preprocessing import StandardScaler, RobustScaler

num_df = df[[c for c in NUMERICAL_FEATURES if c in df.columns]].fillna(0)

std_scaled = pd.DataFrame(
    StandardScaler().fit_transform(num_df), columns=num_df.columns
)
rob_scaled = pd.DataFrame(
    RobustScaler().fit_transform(num_df), columns=num_df.columns
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
std_scaled.boxplot(ax=axes[0], rot=45)
axes[0].set_title("Standard scaling")
rob_scaled.boxplot(ax=axes[1], rot=45)
axes[1].set_title("Robust scaling")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics.pairwise import euclidean_distances, manhattan_distances, cosine_distances

X_std, _ = preprocess_data(df, feature_set="full", scaler="standard")
rng = np.random.default_rng(42)
idx = rng.choice(len(X_std), size=500, replace=False)
X_sample = X_std[idx]

d_l2 = euclidean_distances(X_sample).ravel()
d_l1 = manhattan_distances(X_sample).ravel()
d_cos = cosine_distances(X_sample).ravel()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, d, label in zip(axes, [d_l2, d_l1, d_cos], ["L2", "L1", "Cosine"]):
    ax.hist(d[d > 0], bins=50)
    ax.set_title(f"{label}  \u03bc={d[d>0].mean():.2f}  \u03c3={d[d>0].std():.2f}")
plt.tight_layout()
plt.show()
